# AMADS Coding Notebooks

## offset_fraction

Finding whether `approximate_pulse_match_with_priority_list` can recover the exact
fractional offset that music21 uses for note positions, given only a float.

---

**By (author/s):** Arnav Sayooj

**For:** Attached to the [AMADS code library](https://github.com/music-computing/amads/)

**Licence:** MIT.

In [1]:
import music21
from fractions import Fraction
from amads.time.meter.grid import approximate_pulse_match_with_priority_list

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


## <font color='Green'> music21 Note Offsets

Each note in music21 has an `offset` giving its position within its measure.
When the score is parsed from MusicXML, these offsets are computed from durations
and should be exact rational values, but music21 has them as floats.

`Fraction(n.offset)` converts them back to a fraction. For common musical values
like 0.5 or 0.75, this gives clean results (1/2, 3/4). For triplet values like 1/3
a floating point error may occur.

In [2]:
sc = music21.converter.parse('../amads/music/musicxml/bwv846m15-16.musicxml')

# collect the offset of every note as both a float and a fraction
offset_pairs = []
for n in sc.recurse().notes:
    offset_float = float(n.offset)
    offset_frac = Fraction(n.offset).limit_denominator(10000)
    offset_pairs.append((offset_float, offset_frac))

# get the unique fractions
seen = set()
unique_fracs = []
for _, frac in offset_pairs:
    if frac not in seen:
        seen.add(frac)
        unique_fracs.append(frac)
unique_fracs.sort()

print(f'{len(unique_fracs)} total fractions: ')
for frac in unique_fracs:
    print(frac, '->', float(frac))

16 total fractions: 
0 -> 0.0
1/2 -> 0.5
3/4 -> 0.75
7/8 -> 0.875
1 -> 1.0
5/4 -> 1.25
3/2 -> 1.5
7/4 -> 1.75
2 -> 2.0
5/2 -> 2.5
11/4 -> 2.75
23/8 -> 2.875
3 -> 3.0
7/2 -> 3.5
15/4 -> 3.75
31/8 -> 3.875


## <font color='Green'> Can the approximation function recover these fractions?

`approximate_pulse_match_with_priority_list` takes a float and returns the pulse
(which is the smallest grid unit) that best fits. For example, given 0.8333... it returns
Fraction(1, 6) meaning the note falls on a 1/6-note grid.

We check: is the exact fraction a whole multiple of the returned pulse?

In [3]:
mismatches = []

for frac in unique_fracs:
    # only the sub-beat part matters e.g. 7/4 -> 3/4
    frac_part = frac - int(frac)
    if frac_part == 0:
        continue

    pulse = approximate_pulse_match_with_priority_list(float(frac_part))

    if pulse is None:
        match = False
    else:
        # check that the fraction is a whole multiple of the pulse
        match = (frac_part % pulse == 0)

    if match:
        print('match   ', frac, '  frac_part =', frac_part, '  pulse =', pulse)
    else:
        print('no match', frac, '  frac_part =', frac_part, '  pulse =', pulse)
        mismatches.append(frac)

print()
print(len(mismatches), 'mismatches')

match    1/2   frac_part = 1/2   pulse = 1/2
match    3/4   frac_part = 3/4   pulse = 1/4
match    7/8   frac_part = 7/8   pulse = 1/8
match    5/4   frac_part = 1/4   pulse = 1/4
match    3/2   frac_part = 1/2   pulse = 1/2
match    7/4   frac_part = 3/4   pulse = 1/4
match    5/2   frac_part = 1/2   pulse = 1/2
match    11/4   frac_part = 3/4   pulse = 1/4
match    23/8   frac_part = 7/8   pulse = 1/8
match    7/2   frac_part = 1/2   pulse = 1/2
match    15/4   frac_part = 3/4   pulse = 1/4
match    31/8   frac_part = 7/8   pulse = 1/8

0 mismatches


## <font color='Green'> What distance_threshold is needed?

The default threshold is 0.001. Here we check whether any of the float offsets
actually need a looser threshold to match -- i.e., the float representation of the
exact fraction is far enough from the expected pulse multiple that the function
fails at the default threshold.

In [4]:
import glob

score_files = glob.glob('../amads/music/musicxml/*.musicxml') + glob.glob('../amads/music/musicxml/*.xml')

all_frac_parts = set()
for path in score_files:
    try:
        s = music21.converter.parse(path)
        for n in s.recurse().notes:
            frac = Fraction(n.offset).limit_denominator(10000)
            frac_part = frac - int(frac)
            if frac_part != 0:
                all_frac_parts.add(frac_part)
    except Exception as e:
        print('skipped', path, '-', e)

print(len(all_frac_parts), 'unique sub-beat fractions across all scores')
print()

# for each fraction, find the minimum threshold that makes the approximation work
for frac_part in sorted(all_frac_parts):
    found = False
    for threshold in [0.001, 0.01, 0.05, 0.1]:
        pulse = approximate_pulse_match_with_priority_list(float(frac_part), distance_threshold=threshold)
        if pulse is not None and frac_part % pulse == 0:
            print(frac_part, '  min threshold =', threshold, '  pulse =', pulse)
            found = True
            break
    if not found:
        print(frac_part, '  NO MATCH')

4 unique sub-beat fractions across all scores

1/4   min threshold = 0.001   pulse = 1/4
1/2   min threshold = 0.001   pulse = 1/2
3/4   min threshold = 0.001   pulse = 1/4
7/8   min threshold = 0.001   pulse = 1/8


## <font color='Green'> Triplet fractions

<font color=''>The scores above only have power of 2 denominators (1/2, 1/4, 1/8), so they don't
test the harder case. Triplets produce offsets with denominators of 3, 6, or 12,
which don't convert cleanly to binary floats.

We test those directly by constructing the fractions we'd expect to see in a triplet passage.

In [5]:
triplet_fracs = [
    Fraction(1, 3),   # first triplet eighth
    Fraction(2, 3),   # second triplet eighth
    Fraction(1, 6),   # first triplet sixteenth
    Fraction(5, 6),   # fifth triplet sixteenth
    Fraction(1, 12),  # triplet thirty-second
]

for exact in triplet_fracs:
    f = float(exact)
    recovered = Fraction(f).limit_denominator(10000)
    pulse = approximate_pulse_match_with_priority_list(f)

    if pulse is not None and recovered % pulse == 0:
        print('match   ', exact, '  float =', round(f, 10), '  recovered =', recovered, '  pulse =', pulse)
    else:
        print('no match', exact, '  float =', round(f, 10), '  recovered =', recovered, '  pulse =', pulse)

match    1/3   float = 0.3333333333   recovered = 1/3   pulse = 1/3
match    2/3   float = 0.6666666667   recovered = 2/3   pulse = 1/3
match    1/6   float = 0.1666666667   recovered = 1/6   pulse = 1/6
match    5/6   float = 0.8333333333   recovered = 5/6   pulse = 1/6
match    1/12   float = 0.0833333333   recovered = 1/12   pulse = 1/12
